# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors: Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the FAIR² clinical oncology dataset using the `mlcroissant` library. Each dataset entity is referenced by its unique `@id`, and analysis is performed in a modular and reproducible way.

### Dataset Source

FAIR² Croissant schema:
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`. We will use the Croissant schema URL to instantiate the dataset.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
# The metadata is available as an object (not a dict)
meta = dataset.metadata

print(f"{meta.name}: {meta.description}")

## 2. Data Overview

Let's enumerate all record sets and their fields (referenced by `@id`). This helps us understand the structure and allows us to reference entities correctly in later steps.

In [ ]:
# Discover available record sets (@id and names) and their fields

if hasattr(meta, 'record_sets'):
    record_sets = meta.record_sets
elif hasattr(meta, 'record_set'):
    record_sets = meta.record_set
else:
    record_sets = []

if not record_sets:
    # Sometimes record sets are embedded in the dataset in a different field.
    # We will try to list record sets using the dataset API as a fallback.
    try:
        record_sets = dataset.record_sets
    except AttributeError:
        record_sets = []

print("Record Sets in dataset:")

all_record_set_ids = []
for rs in dataset.record_sets:
    print(f"  @id: {rs.id:70}  | name: {rs.name}")
    all_record_set_ids.append(rs.id)
    print("    Fields:")
    for f in rs.fields:
        # Show field @id, name, and dataType (if present)
        type_str = getattr(f, 'data_type', None)
        print(f"    - @id: {f.id:60} | name: {f.name:30} | dataType: {type_str}")


## 3. Data Extraction

Load data from each record set into a DataFrame for analysis, using the `@id` of the record set and fields as discovered above.

In [ ]:
# Extract data from each record set into pandas DataFrames using record set @id
dfs = {}
record_set_ids = all_record_set_ids

print("\nLoading record sets as pandas DataFrames:")
for rsid in record_set_ids:
    recs = list(dataset.records(record_set=rsid))  # This returns a list of dicts
    print(f"  - {rsid}: {len(recs)} records")
    dfs[rsid] = pd.DataFrame(recs)

# Pick the main record set for demonstration (choose the largest or best match by content)
if dfs:
    main_record_set_id = max(dfs.keys(), key=lambda k: dfs[k].shape[1])  # pick largest (by columns)
    print("\nMain record set for EDA:", main_record_set_id)
    print("Columns:", dfs[main_record_set_id].columns.tolist())
    dfs[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

Let's perform common data processing steps: 
- Filter by a numeric variable (e.g., age or interval)
- Normalize this variable
- Group by an attribute (e.g., sex or tumor site), referencing all columns by their `@id`.

Consult the field list above for the relevant `@id`s.

In [ ]:
# For reproducibility, show all columns again
df = dfs[main_record_set_id]
print("Columns in main DataFrame:")
for col in df.columns:
    print(col)
# Select a numeric field by @id (example: 'age', 'interval_first_second_years', etc.)
# Example: use a substring match to guess 'age' or a numeric interval field

import numpy as np
import re

# Heuristically pick a numeric column (e.g., age, interval, tumor_size, etc.)
numeric_col = None
for c in df.columns:
    col_lower = c.lower()
    if re.search(r'age|interval|years|size|count|number|duration|length', col_lower):
        # Check if this column has numeric dtype or can be converted
        try:
            if pd.to_numeric(df[c], errors='coerce').notnull().sum() > 0:
                numeric_col = c
                break
        except Exception:
            continue

if numeric_col is None:
    # Fallback: pick first numeric column
    for c in df.columns:
        try:
            if pd.to_numeric(df[c], errors='coerce').notnull().sum() > 0:
                numeric_col = c
                break
        except Exception:
            continue

if numeric_col is None:
    raise ValueError("Could not find a numeric column for EDA.")

print(f"Selected numeric field (@id): {numeric_col}")
# Make sure the field is numeric
df[numeric_col] = pd.to_numeric(df[numeric_col], errors='coerce')

# Filtering: remove rows where the value is missing or <= threshold
thresh = df[numeric_col].quantile(0.25) if df[numeric_col].notnull().sum() > 0 else 0
filtered_df = df[df[numeric_col] > thresh].copy()
print(f"Filtered records with {numeric_col} > {thresh:.2f} (25th percentile): {len(filtered_df)} rows")
print(filtered_df[[numeric_col]].head())

# Normalize
filtered_df[numeric_col + "_normalized"] = (filtered_df[numeric_col] - filtered_df[numeric_col].mean()) / filtered_df[numeric_col].std()
print(f"\nNormalized {numeric_col}:")
print(filtered_df[[numeric_col, numeric_col + "_normalized"]].head())

# Group by another field that looks categorical (e.g. sex, tumor location, MSI status, etc.)
group_col = None
possible_group_keys = ['sex', 'gender', 'msi', 'subtype', 'location', 'site', 'status', 'histology']
for candidate in df.columns:
    if any(kw in candidate.lower() for kw in possible_group_keys):
        group_col = candidate
        break
if group_col:
    print(f"\nGrouping by: {group_col}")
    grouped_df = filtered_df.groupby(group_col)[numeric_col].mean()
    print("\nMeans by group:")
    print(grouped_df)
else:
    print("\nNo suitable group column found to group by.")

## 5. Visualization

Let's visualize the distribution of the selected numeric field and compare groups if a grouping variable was found.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,4))
sns.histplot(filtered_df[numeric_col], kde=True, bins=15, color='skyblue')
plt.title(f"Distribution of {numeric_col}")
plt.xlabel(numeric_col)
plt.ylabel("Count")
plt.show()

# Compare groups if categorical field exists
if group_col:
    plt.figure(figsize=(8,5))
    sns.boxplot(x=group_col, y=numeric_col, data=filtered_df)
    plt.title(f"{numeric_col} by {group_col}")
    plt.show()

## 6. Conclusion

We have demonstrated robust, `@id`-referenced exploration of a Croissant-packaged clinical oncology dataset using `mlcroissant`. Key steps included discovering record set and field `@id`s, loading data into DataFrames, filtering, normalization, group-wise comparison, and visualization.

- Data was filtered and normalized based on the selected numeric field (`@id`: {numeric_col}).
- Key distributions and group comparisons were visualized.
- Analysis is fully reproducible and adaptable to any Croissant dataset by referencing entity `@id`s directly.

Next steps: explore additional variables, build predictive models, or combine other record sets for advanced analytics.